In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp, col, concat_ws, trim, coalesce, md5
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType,
    DecimalType, DoubleType
)
import logging
import sys
import datetime
import os

# --- Setup for Logger (Standard Python logging) ---
# This logger setup will print messages to the console.
# In a production environment, you'd integrate with your system's logging.
loggerObject = logging.getLogger(__name__)
loggerObject.setLevel(logging.INFO) # Set the logging level (INFO, DEBUG, WARNING, ERROR, CRITICAL)

# Create a console handler if not already present
if not loggerObject.handlers:
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    loggerObject.addHandler(handler)
# --- End Logger Setup ---

# --- Spark Session Placeholder ---
# This 'spark' variable will be defined in the __main__ block for standalone execution.
# For actual execution in a Databricks/Spark environment, it would typically be pre-defined.
spark = None
# --- End Spark Session Placeholder ---


def merge_into_stage(source_df, primary_key, source_table, target_table, is_merge_enabled='N'):
    """
    Merges data from a source DataFrame into a Delta target table using audit columns.

    This function performs the following steps:
    1. Adds audit columns (`is_deleted`, `row_inserted_time`, `row_updated_time`) to the source data.
    2. Identifies new or updated records by comparing with the source data with the target table.
    3. Performs an upsert (MERGE) into the target Delta table.
    4. Identifies deleted rows (present in target but not in source) and:
        - Inserts them into a deleted rows audit table. Expects the deleted_rows table to be present.
        - Physically deletes them from the target table.

    Parameters:
        source_df (DataFrame): Dataframe with curated source data that is ready to be pushed to target.
        primary_key (str): The column used as a unique identifier for matching records. It can take any number of columns separated by a comma.
        source_table (str): The source table name in 'catalog.schema.table' format.
        target_table (str): The target table name in 'catalog.schema.table' format.
        is_merge_enabled (str): 'Y' to enable merge, 'N' to perform overwrite. Defaults to 'N'.

    Raises:
        Exception: If any error occurs during processing.
    """

    # Ensure spark and loggerObject are accessible (global or passed)
    global spark # If spark is not passed as an argument, ensure it's globally available
    global loggerObject # If loggerObject is not passed as an argument, ensure it's globally available

    try:
        loggerObject.info(f"Starting merge_into_stage function.")
        loggerObject.info(f"Input Parameters: "
                          f"source_table='{source_table}', "
                          f"target_table='{target_table}', "
                          f"primary_key='{primary_key}', "
                          f"is_merge_enabled='{is_merge_enabled}'")
        loggerObject.info(f"Source DataFrame schema: {source_df.schema}")
        loggerObject.info(f"Source DataFrame count: {source_df.count()}")


        if is_merge_enabled == 'N':
            # Step 1: Write the dataframe in overwrite mode
            loggerObject.info(f"Overwrite mode selected for table {target_table}.")
            loggerObject.info(f"Overwrite for table {target_table} started.")
            source_df.write.format("delta") \
                .mode("overwrite") \
                .option("mergeSchema", "true") \
                .saveAsTable(target_table)
            loggerObject.info(f"Overwrite for table {target_table} completed.")
        else:
            loggerObject.info(f"Merge mode selected for table {target_table}.")
            # Note: Multi-table transactions are not supported as of now. The try block is used here only to catch exceptions here.

            # Step 1: Get current timestamp once
            ts = spark.sql("SELECT current_timestamp() AS now").collect()[0]["now"]
            ts_str = ts.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]  # need to convert to string for sql.
            loggerObject.info(f"Current timestamp (ts_str): {ts_str}")

            # Create source temp view for EXCEPT logic based on source_df (audit columns are not added yet)
            table_name = source_table.split(".")[-1] # This assumes source_table is like catalog.schema.tableName
            source_view = f"{table_name}_view"
            source_df.createOrReplaceTempView(source_view)
            loggerObject.info(f"Created source temporary view: {source_view}")
            # Ensure the view is not empty before trying to get schema
            if spark.catalog.tableExists(source_view) and spark.table(source_view).limit(1).count() > 0:
                loggerObject.info(f"Source view schema: {spark.table(source_view).schema}")
            else:
                loggerObject.warning(f"Source view '{source_view}' might be empty or not yet materialized. Schema not available.")


            ####################################################
            # Logic to handle target table existence and audit columns
            target_table_exists = False
            try:
                loggerObject.info(f"Attempting to access target table: {target_table}")
                spark.table(target_table).limit(1).collect() # Try to read a row to confirm existence and accessibility
                target_table_exists = True
                loggerObject.info(f"Target table {target_table} exists and is accessible.")
            except Exception as e:
                loggerObject.warning(f"Target table {target_table} does not exist or is not accessible. Error: {e}")
                # If target table doesn't exist, ALTER TABLE will fail.
                # The assumption here is that the initial `target_table` is created outside this function
                # or will be created implicitly by the first `MERGE WHEN NOT MATCHED THEN INSERT`.
                # If it's truly missing, the MERGE will create it IF it has an INSERT clause
                # and delta.auto.create.table.enabled is true.
                # However, the ALTER TABLE for audit columns expects it to exist.
                # For robust local testing, we'll ensure it's created before the call.

            # Only attempt ALTER TABLE if the table exists (or was just created implicitly)
            # The original code always tried ALTER, which would fail if the table was genuinely absent.
            # In a real pipeline, the target table is often pre-created.
            if target_table_exists:
                loggerObject.info(f"Checking and adding audit columns to {target_table}.")
                current_target_cols = [c.lower() for c in spark.table(target_table).columns]
                audit_cols_to_add = []
                if "is_deleted" not in current_target_cols: audit_cols_to_add.append("is_deleted INT")
                if "row_inserted_time" not in current_target_cols: audit_cols_to_add.append("row_inserted_time TIMESTAMP")
                if "row_updated_time" not in current_target_cols: audit_cols_to_add.append("row_updated_time TIMESTAMP")

                if audit_cols_to_add:
                    alter_sql = f"""
                        ALTER TABLE {target_table} ADD COLUMNS ({", ".join(audit_cols_to_add)})
                    """
                    loggerObject.info(f"Executing ALTER TABLE to add audit columns: {alter_sql}")
                    spark.sql(alter_sql)
                    loggerObject.info(f"Audit columns added to {target_table}.")
                else:
                    loggerObject.info(f"All audit columns already present in {target_table}.")
            else:
                loggerObject.warning(f"Target table {target_table} did not exist initially. "
                                      "Skipping explicit ALTER TABLE for audit columns. "
                                      "They will be added by the MERGE's INSERT clause if target schema is compatible.")
            #####################################################

            # Get columns for comparison and merge
            # Re-fetch target table columns after potential ALTER TABLE
            target_table_columns = [col.lower() for col in spark.table(target_table).columns]
            loggerObject.info(f"Target table columns (after audit column check): {target_table_columns}")

            # Determine columns for comparison, excluding audit columns
            base_target_cols = [c for c in spark.table(target_table).columns if c.lower() not in ["dw_insert_dt", "is_deleted", "row_inserted_time", "row_updated_time"]]
            # Filter source_df columns to match target_table's non-audit columns (case-insensitive check)
            common_columns = [col_name for col_name in source_df.columns if col_name.lower() in [bc.lower() for bc in base_target_cols]]
            common_columns_list = ", ".join([f"`{col}`" for col in common_columns])
            loggerObject.info(f"Common columns for merge (base data): {common_columns}")
            loggerObject.info(f"Common columns list for merge SQL: {common_columns_list}")

            # For deletes, we need all target columns that are also in source, including PKs for matching
            target_columns_for_delete_match = spark.table(target_table).columns
            common_columns_delete = [col_name for col_name in source_df.columns if col_name.lower() in [tc.lower() for tc in target_columns_for_delete_match]]
            common_columns_list_for_deletes = ", ".join([f"t.`{col}`" for col in common_columns_delete])
            loggerObject.info(f"Common columns for delete matching: {common_columns_delete}")
            loggerObject.info(f"Common columns list for deletes SQL: {common_columns_list_for_deletes}")


            ###################################################################
            # Identify new and updated rows using MD5 hash comparison
            merge_view = f"{table_name}_merge_view"
            loggerObject.info(f"Creating DataFrames for MD5 hash comparison.")
            source_view_df = spark.sql(f"SELECT {common_columns_list} FROM {source_view}")
            target_table_df = spark.sql(f"SELECT {common_columns_list} FROM {target_table}")
            loggerObject.info(f"Source view DF count for MD5: {source_view_df.count()}")
            loggerObject.info(f"Target table DF count for MD5: {target_table_df.count()}")

            # Create 'mdf_column' column for both DataFrames: cast to string, trim, fillna('*~*'), then concat
            def build_mdf_column(df):
                cols = [trim(coalesce(col(c).cast("string"), lit("*~*"))) for c in df.columns]
                return df.withColumn("mdf_hash_column", md5(concat_ws("||", *cols)))

            source_view_df = build_mdf_column(source_view_df)
            target_table_df = build_mdf_column(target_table_df)
            loggerObject.info(f"MD5 hash columns added to comparison DataFrames.")

            # Perform a left_anti join to find records in source_view_df that are NOT in target_table_df based on MD5 hash
            # This identifies new records AND records where data has changed (thus MD5 hash differs)
            merge_df = source_view_df.alias("s").join(target_table_df.alias("t"), col("s.mdf_hash_column") == col("t.mdf_hash_column"), how="left_anti").select([col(f"s.{c}") for c in common_columns])
            loggerObject.info(f"Identified new/updated records (merge_df) count: {merge_df.count()}")
            ##################################################################

            # Add audit fields to merge_df in preparation to merge to target table
            merge_df_with_audit = merge_df
            if "dw_insert_dt" in target_table_columns:
                merge_df_with_audit = (
                    merge_df_with_audit # Use the already filtered merge_df
                    .withColumn('DW_Insert_Dt', current_timestamp())
                    .withColumn('is_deleted', lit(0))
                    .withColumn('row_inserted_time', current_timestamp())
                    .withColumn('row_updated_time', current_timestamp())
                )
                loggerObject.info(f"Audit columns (including DW_Insert_Dt) added to merge_df_with_audit.")
            else:
                merge_df_with_audit = (
                    merge_df_with_audit # Use the already filtered merge_df
                    .withColumn('is_deleted', lit(0))
                    .withColumn('row_inserted_time', current_timestamp())
                    .withColumn('row_updated_time', current_timestamp())
                )
                loggerObject.info(f"Audit columns added to merge_df_with_audit.")

            merge_df_with_audit.createOrReplaceTempView(merge_view)
            loggerObject.info(f"Created temporary view '{merge_view}' for merge_df_with_audit.")
            loggerObject.info(f"Schema of {merge_view}: {spark.table(merge_view).schema}")

            # Get columns for insert logic
            insert_columns = merge_df_with_audit.columns
            insert_columns_formatted = [f"`{col}`" for col in merge_df_with_audit.columns]
            insert_values = [f"source.`{col}`" for col in insert_columns]
            loggerObject.info(f"Insert columns: {insert_columns}")

            # START: Handle multiple primary key columns + audit columns
            loggerObject.info(f"Preparing primary key and update clauses.")
            pk_cols = [col.strip() for col in primary_key.split(",")]
            loggerObject.info(f"Primary key columns: {pk_cols}")

            excluded_cols = pk_cols + ['is_deleted', 'row_inserted_time', 'row_updated_time']
            if "dw_insert_dt" in target_table_columns:
                excluded_cols.append('DW_Insert_Dt')
            loggerObject.info(f"Excluded columns from update: {excluded_cols}")

            on_clause = " AND ".join([f"trim(upper(source.`{col}`)) = trim(upper(target.`{col}`))" for col in pk_cols])
            loggerObject.info(f"MERGE ON clause: {on_clause}")
            # END: Handle multiple primary key columns

            updatable_columns = [col for col in insert_columns if col not in excluded_cols]
            loggerObject.info(f"Updatable columns: {updatable_columns}")

            update_set_clause = ",\n  ".join(
                [f"target.`{col}` = source.`{col}`" for col in updatable_columns] + [
                    "target.is_deleted = 0",
                    f"target.row_updated_time = cast('{ts_str}' AS TIMESTAMP)"
                ]
            )
            loggerObject.info(f"Update SET clause: {update_set_clause}")

            # Run final MERGE
            loggerObject.info(f"Merging {merge_view} into {target_table}.")
            merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {merge_view} AS source
            ON {on_clause}
            WHEN MATCHED THEN
            UPDATE SET
            {update_set_clause}
            WHEN NOT MATCHED THEN
            INSERT ({", ".join(insert_columns_formatted)})
            VALUES ({", ".join(insert_values)})
            """
            loggerObject.info(f"Full MERGE SQL query:\n{merge_sql}")
            audit_df = spark.sql(merge_sql)

            # Check if audit_df has the expected columns before accessing
            if "num_updated_rows" in audit_df.columns and "num_inserted_rows" in audit_df.columns:
                updated_count = audit_df.select("num_updated_rows").collect()[0][0]
                inserted_count = audit_df.select("num_inserted_rows").collect()[0][0]
                loggerObject.info(f"Rows updated: {updated_count}, Rows inserted: {inserted_count}")
            else:
                loggerObject.warning("Audit DataFrame from MERGE operation did not contain 'num_updated_rows' or 'num_inserted_rows'. Cannot log counts.")
            loggerObject.info(f"***************************Merge Operation Completed******************************")

            # Delete Handling

            # Identify deleted rows - that are in target but not in source
            loggerObject.info(f"Delete Handling - Identifying deleted rows (present in target but not in source).")
            # The join condition for anti-join should only use primary keys for matching
            pk_on_clause_for_anti_join = " AND ".join([f"trim(upper(t.`{col}`)) = trim(upper(s.`{col}`))" for col in pk_cols])
            loggerObject.info(f"Delete identification ON clause: {pk_on_clause_for_anti_join}")

            # Select all columns from target for deleted_df, but match on PKs from source
            # Here, common_columns_list_for_deletes should ideally select all columns from target
            # that are needed for the deleted_rows table, and that can be matched with source.
            # The log's `common_columns_list_for_deletes` is good.
            deleted_df_without_all_columns = spark.sql(f"""
                SELECT {common_columns_list_for_deletes}, 1 AS is_deleted, t.row_inserted_time  -- keep existing row_inserted_time value
                FROM {target_table} t
                LEFT ANTI JOIN {source_view} s
                ON {pk_on_clause_for_anti_join}
            """)
            deleted_df = deleted_df_without_all_columns.withColumn('row_updated_time', lit(ts))
            loggerObject.info(f"Identified deleted rows count (before filtering if empty): {deleted_df.count()}")

            # Check if any rows were deleted
            if deleted_df.count() == 0:
                loggerObject.info(f"No deleted rows found for {target_table}. Skipping deletion sync.")
            else:
                # Step 3: Create temp view for deleted rows
                deleted_rows_view = f"{table_name}_deleted_rows_view"
                deleted_df.createOrReplaceTempView(deleted_rows_view)
                loggerObject.info(f"Created temporary view '{deleted_rows_view}' for {deleted_df.count()} deleted rows.")
                loggerObject.info(f"Schema of {deleted_rows_view}: {spark.table(deleted_rows_view).schema}")

                # Step 4: Create deleted_rows tables in advance
                #########################################################################################
                abstract_target_table_name = target_table.split(".")[-1]
                deleted_rows_table = f"{target_table}_deleted_rows"

                # Attempt to get the location of the target table
                target_table_location = None
                try:
                    target_table_location = spark.sql(f"DESCRIBE DETAIL {target_table}").select("location").collect()[0]["location"]
                    # Ensure path is local file system compatible for local SparkSession
                    if target_table_location.startswith("dbfs:/"): # Databricks specific path
                         target_table_location = target_table_location.replace("dbfs:/", "file:///") # For local testing
                    elif not target_table_location.startswith("file:///"):
                         target_table_location = "file:///" + target_table_location # Assume local path if not explicitly file://

                    delete_table_location = target_table_location.replace(abstract_target_table_name, f"{abstract_target_table_name}_deleted_rows")

                    # Ensure the directory exists for the deleted rows Delta table
                    # Remove "file:///" prefix for os.makedirs, then add back for SQL
                    dir_path = delete_table_location.replace("file:///", "")
                    os.makedirs(dir_path, exist_ok=True)


                    loggerObject.info(f"Target table location: {target_table_location}")
                    loggerObject.info(f"Calculated deleted rows table location: {delete_table_location}")

                    create_deleted_table_sql = f"""
                        CREATE TABLE IF NOT EXISTS {deleted_rows_table}
                        USING DELTA
                        LOCATION '{delete_table_location}'
                        AS SELECT * FROM {target_table}
                        WHERE 1 = 0
                    """
                    loggerObject.info(f"Creating deleted rows table if not exists. SQL:\n{create_deleted_table_sql}")
                    spark.sql(create_deleted_table_sql)
                    loggerObject.info(f"Deleted rows table '{deleted_rows_table}' ensured to exist.")
                except Exception as e:
                    loggerObject.error(f"Failed to get target table location or create deleted rows table. Error: {e}", exc_info=True)
                    raise # Re-raise the exception as this is critical

                #########################################################################################

                # Step 5: Insert deleted rows into deleted_rows table
                loggerObject.info(f"Inserting deleted rows into {deleted_rows_table}.")
                merge_on_deleted = " AND ".join([f"trim(upper(t.`{col}`)) = trim(upper(d.`{col}`))" for col in pk_cols])
                insert_deleted_sql = f"""
                    MERGE INTO {deleted_rows_table} t
                    USING {deleted_rows_view} d
                    ON {merge_on_deleted}
                    WHEN NOT MATCHED THEN
                    INSERT *
                """
                loggerObject.info(f"MERGE SQL for inserting deleted rows:\n{insert_deleted_sql}")
                spark.sql(insert_deleted_sql)
                loggerObject.info(f"Deleted rows inserted into {deleted_rows_table}.")

                # Step 6: Physically delete rows from target table
                loggerObject.info(f"Deleting identified rows from {target_table}.")
                delete_from_target_sql = f"""
                    MERGE INTO {target_table} t
                    USING {deleted_rows_view} d
                    ON {merge_on_deleted}
                    WHEN MATCHED THEN
                    DELETE
                """
                loggerObject.info(f"MERGE SQL for deleting from target table:\n{delete_from_target_sql}")
                spark.sql(delete_from_target_sql)
                loggerObject.info(f"Deleted rows physically removed from {target_table}.")

                # Step 7: Delete already re-inserted rows from deleted_rows_table
                loggerObject.info(f"Cleaning up {deleted_rows_table} by removing records that have reappeared in {target_table}.")
                # This merge_on_deleted uses the PKs to match target (current valid records) against the deleted_rows_table
                delete_reinserted_sql = f"""
                    MERGE INTO {deleted_rows_table} t
                    USING {target_table} d
                    ON {merge_on_deleted}
                    WHEN MATCHED THEN
                    DELETE
                """
                loggerObject.info(f"MERGE SQL for cleaning deleted_rows_table:\n{delete_reinserted_sql}")
                spark.sql(delete_reinserted_sql)
                loggerObject.info(f"Re-inserted records removed from {deleted_rows_table}.")

    except Exception as e:
        loggerObject.error(f"Error during merge_to_stage for table {target_table}: {e}", exc_info=True) # exc_info=True logs the full traceback
        raise # Re-raise the exception after logging


# --- Main execution block for manual variable population and testing ---
if __name__ == "__main__":
    loggerObject.info("--- Starting standalone execution block ---")

    # 1. Initialize a local SparkSession
    spark = SparkSession.builder \
        .appName("LocalMergeTestWithLogData") \
        .master("local[*]") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.sql.warehouse.dir", "spark-warehouse") \
        .getOrCreate()

    loggerObject.info("Local SparkSession initialized.")
    loggerObject.info(f"Spark version: {spark.version}")

    # Clean up previous test runs (optional, but good for consistent testing)
    spark.sql("DROP TABLE IF EXISTS xliidw_qa_lpl.staging_xlprof_genius.AccountRevenueLink")
    spark.sql("DROP TABLE IF EXISTS xliidw_qa_lpl.staging_xlprof_genius.AccountRevenueLink_deleted_rows")
    spark.sql("DROP DATABASE IF EXISTS xliidw_qa_lpl.staging_xlprof_genius CASCADE")
    spark.sql("DROP DATABASE IF EXISTS xliidw_qa_lpl.us_genius CASCADE") # For source_table logical view

    # Create catalogs/schemas if they don't exist for local testing
    spark.sql("CREATE DATABASE IF NOT EXISTS xliidw_qa_lpl.staging_xlprof_genius")
    spark.sql("CREATE DATABASE IF NOT EXISTS xliidw_qa_lpl.us_genius")


    # 2. Define your test variables based on the log
    test_primary_key = "PFKStatsLedgerSeq,PFKRphSeq"
    test_source_table_name = "xliidw_qa_lpl.us_genius.ZARHDF00" # Logical name for source view
    test_target_table_name = "xliidw_qa_lpl.staging_xlprof_genius.AccountRevenueLink"

    # Define the schema based on the log output
    common_schema_cols = [
        StructField("PFKStatsLedgerSeq", DecimalType(10,0), True),
        StructField("PFKRphSeq", DecimalType(6,0), True),
        StructField("RevenuePostingSeq", DecimalType(10,0), True),
        StructField("RevenueCode", StringType(), True),
        StructField("AcctingAmt", DoubleType(), True),
        StructField("BaseAmt", DoubleType(), True),
        StructField("AcctingPeriod", StringType(), True),
        StructField("RphFlag1", StringType(), True),
        StructField("RphFlag2", StringType(), True),
        StructField("RphFlag3", StringType(), True),
        StructField("Term", StringType(), True),
        StructField("DW_Insert_Dt", TimestampType(), True) # This is part of the source in your log
    ]
    source_schema = StructType(common_schema_cols)

    # Target schema includes audit columns
    target_schema = StructType(common_schema_cols + [
        StructField("is_deleted", IntegerType(), True),
        StructField("row_inserted_time", TimestampType(), True),
        StructField("row_updated_time", TimestampType(), True)
    ])

    # Create sample data for source_df
    # Data to be inserted/updated
    source_data = [
        (100, 1, 1000, "REV1", 100.50, 100.00, "202301", "F1A", "F2A", "F3A", "T1", datetime.datetime.now()), # New record
        (101, 2, 1001, "REV2", 200.75, 200.00, "202302", "F1B", "F2B", "F3B", "T2", datetime.datetime.now()), # Updated record (value will change)
        (102, 3, 1002, "REV3", 300.25, 300.00, "202303", "F1C", "F2C", "F3C", "T3", datetime.datetime.now())  # Unchanged record (should not be updated)
    ]
    test_source_df = spark.createDataFrame(data=source_data, schema=source_schema)
    loggerObject.info("Test_source_df created:")
    test_source_df.show()

    # Create initial target table data (simulating existing data)
    target_data_initial = [
        # This record exists in source, but 'AcctingAmt' will be updated
        (101, 2, 1001, "REV2", 200.00, 200.00, "202302", "F1B", "F2B", "F3B", "T2", datetime.datetime.now(), 0, datetime.datetime.now(), datetime.datetime.now()),
        # This record is in source and target, but won't change
        (102, 3, 1002, "REV3", 300.25, 300.00, "202303", "F1C", "F2C", "F3C", "T3", datetime.datetime.now(), 0, datetime.datetime.now(), datetime.datetime.now()),
        # This record exists only in target, so it should be deleted
        (103, 4, 1003, "REV4", 400.00, 400.00, "202304", "F1D", "F2D", "F3D", "T4", datetime.datetime.now(), 0, datetime.datetime.now(), datetime.datetime.now())
    ]
    initial_target_df = spark.createDataFrame(data=target_data_initial, schema=target_schema)

    # Save initial target data to the Delta table (this also creates the table)
    loggerObject.info(f"Creating/overwriting initial target Delta table: {test_target_table_name}")
    initial_target_df.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(test_target_table_name)
    loggerObject.info(f"Initial target table {test_target_table_name} created.")
    loggerObject.info(f"Content of {test_target_table_name} after initial write:")
    spark.table(test_target_table_name).show()


    # 3. Call the function with your manually populated variables
    loggerObject.info("--- Calling merge_into_stage with is_merge_enabled='Y' ---")
    try:
        merge_into_stage(
            source_df=test_source_df,
            primary_key=test_primary_key,
            source_table=test_source_table_name,
            target_table=test_target_table_name,
            is_merge_enabled='Y'
        )
        loggerObject.info("merge_into_stage function completed successfully (Merge mode).")

        loggerObject.info(f"Content of {test_target_table_name} after merge operation:")
        spark.table(test_target_table_name).show()

        deleted_rows_table = f"{test_target_table_name}_deleted_rows"
        loggerObject.info(f"Content of {deleted_rows_table} after merge operation:")
        try:
            spark.table(deleted_rows_table).show()
        except Exception as e:
            loggerObject.warning(f"Could not show {deleted_rows_table}, perhaps it's empty or doesn't exist yet: {e}")

    except Exception as e:
        loggerObject.error(f"Execution of merge_into_stage (Merge mode) failed: {e}")

    # You can also test the 'N' (overwrite) case if needed, but the primary focus was on the merge/delete error.
    # loggerObject.info("--- Preparing for overwrite test ---")
    # source_data_overwrite = [
    #     (200, 5, 2000, "REV5", 500.00, 500.00, "202401", "F1E", "F2E", "F3E", "T5", datetime.datetime.now())
    # ]
    # test_source_df_overwrite = spark.createDataFrame(data=source_data_overwrite, schema=source_schema)
    # loggerObject.info("Test_source_df for overwrite created:")
    # test_source_df_overwrite.show()

    # loggerObject.info("--- Calling merge_into_stage with is_merge_enabled='N' (overwrite) ---")
    # try:
    #     merge_into_stage(
    #         source_df=test_source_df_overwrite,
    #         primary_key=test_primary_key, # Primary key not strictly used in overwrite mode
    #         source_table=test_source_table_name,
    #         target_table=test_target_table_name,
    #         is_merge_enabled='N'
    #     )
    #     loggerObject.info("merge_into_stage function (overwrite) completed successfully.")
    #     loggerObject.info(f"Content of {test_target_table_name} after overwrite operation:")
    #     spark.table(test_target_table_name).show()
    # except Exception as e:
    #     loggerObject.error(f"Execution of merge_into_stage (overwrite) failed: {e}")


    # 4. Stop the SparkSession
    spark.stop()
    loggerObject.info("Local SparkSession stopped.")
    loggerObject.info("--- End standalone execution block ---")